In [ ]:
import sqlite3
import os
import numpy as np
import rasterio

from datetime import date, time, datetime, timedelta

YEAR = 2025
ROOT = "/Users/jungtaeuk/ships/"
DB_PATH = os.path.join(ROOT, "ships.db")
TIF_DIR = "/Volumes/SAMSUNG/JPSS-2_VIIRS/GeoTIFF/"

def get_ship_lonlat_by_img_dt(db_path: str, img_dt: str):
    """
    img_dt: 'YYYY-MM-DD HH:MM:SS'
    return: [(lon, lat), ...]
    """
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    cur.execute("""
        SELECT Lon, Lat
        FROM ships_dynamic
        WHERE IMG_DT = ?
          AND ITPL > 0
    """, (img_dt,))

    rows = cur.fetchall()
    conn.close()

    # [(lon, lat), ...]
    return [(lon, lat) for lon, lat in rows]

def get_unique_img_dt_list(db_path: str):
    """
    return: ['YYYY-MM-DD HH:MM:SS', ...] (시간순 정렬)
    """
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    cur.execute("""
        SELECT DISTINCT IMG_DT
        FROM ships_dynamic
        ORDER BY IMG_DT ASC
    """)

    img_dt_list = [row[0] for row in cur.fetchall()]
    conn.close()

    return img_dt_list

def _mapped_date()->dict: # 위성 영상의 날짜 (001 ~ 365) -> datetime 객체로 변환
    start_date = date(YEAR, 1, 1)
    doy_to_date = {doy: (start_date + timedelta(days=doy - 1)).isoformat() for doy in range(1, 366)}
    return doy_to_date

def parse_key(fname :str)->tuple[int, int]:
    base = fname.replace(".tif", "")
    date, hhmm, _ = base.split("_")
    return (int(date[1:]), int(hhmm))

def tif_name_to_datetime(
    tif_name: str,
    mapped_date_dict: dict[int, str]
) -> datetime:
    """
    tif_name 예: '2025001_1754_021.tif'
    return: datetime(2025-01-01 17:54:00)
    """
    base = tif_name.replace(".tif", "")
    date_part, hhmm, *_ = base.split("_")

    doy = int(date_part[-3:])   # 001
    hour = int(hhmm[:2])
    minute = int(hhmm[2:])

    date_str = mapped_date_dict[doy]  # '2025-01-01'

    return datetime(
        year=YEAR,
        month=int(date_str[5:7]),
        day=int(date_str[8:10]),
        hour=hour,
        minute=minute,
        second=0
    )

def find_neighbor_tifs(
    img_dt: str,
    tif_name_list: list[str],
    mapped_date_dict: dict[int, str],
    window_min: int = 6
) -> list[str]:
    """
    target_tif 기준 ±window_min 분 내에 포함되는 tif_name 반환
    """
    img_dt_datetime = datetime.strptime(img_dt, "%Y-%m-%d %H:%M:%S").replace(microsecond=0)
    dt_min = img_dt_datetime - timedelta(minutes=window_min)

    selected = []

    for tif in tif_name_list:
        try:
            tif_dt = tif_name_to_datetime(tif, mapped_date_dict)
            if dt_min <= tif_dt:
                selected.append(tif)
        except Exception:
            continue  # 형식 안 맞는 파일명 등

    # 시간순 정렬 (중요)
    selected.sort(
        key=lambda x: tif_name_to_datetime(x, mapped_date_dict)
    )

    return selected

def workhorse(tif_path : str, img_dt : str):
    ship_lonlat_list = get_ship_lonlat_by_img_dt(DB_PATH, img_dt)
    with rasterio.open(tif_path) as ds:
        R = ds.read(1, masked=True)          # (H,W) radiance (masked array)
        R = np.tan((np.pi / 2) * R) * 1e-9
        H, W = R.shape
        C = np.zeros((H, W), dtype=np.uint16)
        for lon, lat in ship_lonlat_list:
            try:
                r, c = ds.index(lon, lat)
                if 0 <= r < H and 0 <= c < W:
                    C[r, c] += 1
            except Exception:
                pass  # 영상 밖/좌표 문제 등
        valid = ~np.ma.getmaskarray(R) & np.isfinite(R)
        x = np.asarray(R[valid], dtype=np.float64)     # radiance
        y = C[valid].astype(np.int32)                  # count
        return x, y, C


In [ ]:
img_dt_list = get_unique_img_dt_list(DB_PATH)
mapped_date_dict = _mapped_date()
tif_list = [name for name in os.listdir(TIF_DIR) if (not name.startswith('.') and len(name) > 15)]
results = []
for i, img_dt in enumerate(img_dt_list):
    if i == len(tif_list):
        break
    if img_dt is not None:
        tif_name = find_neighbor_tifs(img_dt, tif_list, mapped_date_dict, 6)[0]
        x, y, C = workhorse(os.path.join(TIF_DIR, tif_name),img_dt)
        results.append({"img_dt": img_dt, "x": x, "y": y, "C": C})

In [ ]:
print(len(results))

In [ ]:
import matplotlib.pyplot as plt

for data in results:#[:1]:
    img_dt = data["img_dt"]
    x = np.log(data["x"])
    y = data["y"]
    plt.figure(figsize=(6, 4))
    plt.scatter(y, x, s=2, alpha=0.3)
    plt.xlabel("Ship count per pixel")
    plt.ylabel("Radiance (log-scale)")
    plt.title(f"Radiance vs Ship Count @ {img_dt}")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# x: log-radiance (또는 radiance)
# y: count

# 1) masked -> ndarray로 변환 (mask는 NaN으로)
x2 = np.asarray(np.ma.filled(x, np.nan), dtype=np.float64)
y2 = np.asarray(np.ma.filled(y, np.nan), dtype=np.float64)

# 2) 유효값만 남기기 (NaN/Inf 제거)
valid = np.isfinite(x2) & np.isfinite(y2)

# (옵션) count는 음수/이상치 방지
valid &= (y2 >= 0)

x2 = x2[valid]
y2 = y2[valid].astype(np.int32)

# 3) bin 만들기
count_bins = np.arange(y2.max() + 2) - 0.5
rad_bins   = np.linspace(x2.min(), x2.max(), 100)

# 4) histogram2d
H, xedges, yedges = np.histogram2d(
    y2, x2,
    bins=[count_bins, rad_bins]
)

plt.figure(figsize=(6, 5))
plt.imshow(
    H.T,
    origin="lower",
    aspect="auto",
    norm=LogNorm(),
    extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]]
)
plt.xlabel("Ship count per pixel")
plt.ylabel("Radiance (log scale)" if "log" in str(type(x)) else "Radiance")
plt.title(f"Radiance vs Ship Count Heatmap @ {img_dt}")
plt.colorbar(label="Number of pixels (log)")
plt.tight_layout()
plt.show()